In [171]:
!python -V
!pip install mlflow

Python 3.12.0


In [172]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import os

In [173]:
a=os.curdir
os.listdir(a)

['delivery_forecasting.ipynb', 'supply_chain_large.csv']

In [174]:
df=pd.read_csv("supply_chain_large.csv")

In [175]:
#5 premières lignes
df.head()
# #nombre de lignes du df
# len(df)
# #nombre d'element dans le df
# df.size
# #nombre de ligne et de colonnes
# df.shape

,order_id,product_id,origin_warehouse,destination_city,order_date,ship_mode,carrier,weight_kg,distance_km,expected_delay_days
0,100000,P052,Rouen,Bordeaux,2024-11-23,Standard,GLS,2.98,983,4
1,100001,P093,Nantes,Nice,2025-06-16,Express,Colissimo,12.82,724,3
2,100002,P015,Rouen,Marseille,2024-08-07,Standard,DHL,14.11,392,4
3,100003,P072,Rouen,Paris,2024-12-18,Express,Chronopost,12.38,380,2
4,100004,P061,Paris,Lyon,2025-03-30,Express,DHL,18.77,373,2


In [176]:
#vérifier s'il y a une valeur manquante
df.isnull().values.any()

np.False_

In [177]:
df.columns

Index(['order_id', 'product_id', 'origin_warehouse', 'destination_city',
       'order_date', 'ship_mode', 'carrier', 'weight_kg', 'distance_km',
       'expected_delay_days'],
      dtype='object')

In [178]:
df.dtypes

order_id                 int64
product_id              object
origin_warehouse        object
destination_city        object
order_date              object
ship_mode               object
carrier                 object
weight_kg              float64
distance_km              int64
expected_delay_days      int64
dtype: object

In [179]:
a= df.columns
a


Index(['order_id', 'product_id', 'origin_warehouse', 'destination_city',
       'order_date', 'ship_mode', 'carrier', 'weight_kg', 'distance_km',
       'expected_delay_days'],
      dtype='object')

In [180]:
def convert_to_string(df):
    a= df.columns
    for i in list(a):
        print(i)
        # df[f"i"]=df[f"i"].astype(str)
        return type(a)

In [181]:
convert_to_string(df)

order_id


pandas.core.indexes.base.Index

In [182]:
df["product_id"] = df["product_id"].astype(str)
df["product_id"] = df["product_id"].astype(str)


In [183]:
df.dtypes
df.columns

Index(['order_id', 'product_id', 'origin_warehouse', 'destination_city',
       'order_date', 'ship_mode', 'carrier', 'weight_kg', 'distance_km',
       'expected_delay_days'],
      dtype='object')

In [184]:
#séparer le jeu de données en var categorielles et numériques
# categorical=["origin_warehouse","ship_mode","ship_mode"]
categorical=["product_id","destination_city","carrier"]
numerical=["weight_kg"]

In [185]:
#convertir les variables categorielles en str pour faciliter la vectorization
df[categorical]=df[categorical].astype(str)

In [186]:
#concatene les 2 listes categorical et numerical en dictionnaire
train_dicts=df[categorical + numerical].to_dict(orient='records')
train_dicts
#si on veut travailler sur un dictionnaire

[{'product_id': 'P052',
  'destination_city': 'Bordeaux',
  'carrier': 'GLS',
  'weight_kg': 2.98},
 {'product_id': 'P093',
  'destination_city': 'Nice',
  'carrier': 'Colissimo',
  'weight_kg': 12.82},
 {'product_id': 'P015',
  'destination_city': 'Marseille',
  'carrier': 'DHL',
  'weight_kg': 14.11},
 {'product_id': 'P072',
  'destination_city': 'Paris',
  'carrier': 'Chronopost',
  'weight_kg': 12.38},
 {'product_id': 'P061',
  'destination_city': 'Lyon',
  'carrier': 'DHL',
  'weight_kg': 18.77},
 {'product_id': 'P021',
  'destination_city': 'Lyon',
  'carrier': 'Chronopost',
  'weight_kg': 11.13},
 {'product_id': 'P083',
  'destination_city': 'Bordeaux',
  'carrier': 'Chronopost',
  'weight_kg': 18.11},
 {'product_id': 'P087',
  'destination_city': 'Paris',
  'carrier': 'Colissimo',
  'weight_kg': 5.32},
 {'product_id': 'P075',
  'destination_city': 'Bordeaux',
  'carrier': 'DHL',
  'weight_kg': 10.79},
 {'product_id': 'P075',
  'destination_city': 'Marseille',
  'carrier': 'GLS'

In [187]:
a=df[categorical]
b=df[numerical]

# df1=pd.concat([a, b], ignore_index=True)
# df1


In [188]:
encoder = OneHotEncoder(sparse_output=False)  # sparse_output=False pour un tableau dense (lisible)

# Appliquer l'encodage
cat=encoder.fit_transform(df[categorical])
# encoder.get_feature_names_out()
type(cat)
cat

array([[0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 1., 0., 0.]])

In [189]:
#variable explicatives
features = np.hstack((cat, df["weight_kg"].values.reshape(-1, 1)))
len(features)
#variable cible
target=df["expected_delay_days"]
target

0        4
1        3
2        4
3        2
4        2
        ..
99995    2
99996    5
99997    4
99998    5
99999    3
Name: expected_delay_days, Length: 100000, dtype: int64

In [190]:
#separation de jeu de données
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)


In [191]:
lr= LinearRegression()
lr.fit(X_train,y_train)
y_pred=lr.predict(X_test)

In [192]:
print("R² :", r2_score(y_test, y_pred))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("MSE :", mean_squared_error(y_test, y_pred))

R² : 0.18295090383432944
MAE : 0.9690114837646484
MSE : 1.406167239718139
